# Homework #2
## Data Loading and Inspection

In [0]:
df_pitstops = spark.read.csv(
  "/Volumes/gr5069/raw/f1_data/pit_stops.csv",
  header=True,
  inferSchema=True
)

display(df_pitstops)

In [0]:
df_pitstops = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/pit_stops.csv",
    header=True,
    inferSchema=True
)

In [0]:
df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True,
    inferSchema=True
)

df_races = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/races.csv",
    header=True,
    inferSchema=True
)

df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

In [0]:
df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True,
    inferSchema=True
)

df_races = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/races.csv",
    header=True,
    inferSchema=True
)

df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

In [0]:
drivers = df_drivers
races = df_races
results = df_results
pit_stops = df_pitstops

In [0]:
drivers = drivers.withColumn("dob", F.to_date("dob"))
races = races.withColumn("date", F.to_date("date"))
pit_stops = pit_stops.withColumn("milliseconds", F.col("milliseconds").cast("double"))

In [0]:
q1 = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.avg("milliseconds").alias("avg_pit_ms"),
        F.min("milliseconds").alias("fastest_pit_ms"),
        F.max("milliseconds").alias("slowest_pit_ms")
    )
    .join(drivers.select("driverId", "forename", "surname"), "driverId")
    .join(races.select("raceId", "year", "name"), "raceId")
    .orderBy("year", "name", "avg_pit_ms")
)

display(q1)

## Question 1


To answer this question, I use the `pit_stops` dataset, which contains one row for each pit stop event and includes the time spent in milliseconds. I group the data by `raceId` and `driverId` so that each group represents one driver in one race. Then, I compute the average, minimum, and maximum pit stop time for each driver in each race. Finally, I join the result with the `drivers` and `races` datasets to include driver names and race information.


In [0]:
position_col = "positionOrder" if "positionOrder" in results.columns else "position"

avg_pit = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.avg("milliseconds").alias("avg_pit_ms"))
)

q2 = (
    avg_pit
    .join(results.select("raceId", "driverId", position_col), ["raceId", "driverId"])
    .join(drivers.select("driverId", "forename", "surname"), "driverId")
    .join(races.select("raceId", "year", "name"), "raceId")
    .orderBy("year", "name", position_col)
)

display(q2)

## Question 2


To answer this question, I first calculate the average pit stop time for each driver in each race using the `pit_stops` dataset. Then, I join this result with the `results` dataset to obtain each driver's finishing position. Finally, I join with the `drivers` and `races` datasets to include descriptive information and sort the output by race and finishing position.


In [0]:
q3 = (
    drivers
    .withColumn(
        "code_filled",
        F.when(
            F.col("code").isNull() | (F.trim(F.col("code")) == ""),
            F.upper(F.substring(F.col("surname"), 1, 3))
        ).otherwise(F.col("code"))
    )
)

display(q3.select("driverId", "forename", "surname", "code", "code_filled"))

## Question 3


The `drivers` dataset contains a column called `code`, but some values are missing. To fill in missing values, I generate a replacement code using the driver's surname. Specifically, I take the first three letters of the surname and convert them to uppercase. If the original code exists, I keep it; otherwise, I use the generated value.



This ensures that all drivers have a valid code value.

In [0]:
race_driver_age = (
    results
    .select("raceId", "driverId")
    .join(drivers.select("driverId", "forename", "surname", "dob"), "driverId")
    .join(races.select("raceId", "year", "name", "date"), "raceId")
    .withColumn("Age", F.floor(F.months_between(F.col("date"), F.col("dob")) / 12))
)

youngest = (
    race_driver_age
    .withColumn("rn", F.row_number().over(Window.partitionBy("raceId").orderBy("Age")))
    .filter("rn = 1")
    .select(
        "raceId", "year", "name",
        F.concat_ws(" ", "forename", "surname").alias("youngest_driver"),
        F.col("Age").alias("youngest_age")
    )
)

oldest = (
    race_driver_age
    .withColumn("rn", F.row_number().over(Window.partitionBy("raceId").orderBy(F.col("Age").desc())))
    .filter("rn = 1")
    .select(
        "raceId",
        F.concat_ws(" ", "forename", "surname").alias("oldest_driver"),
        F.col("Age").alias("oldest_age")
    )
)

q4 = youngest.join(oldest, "raceId").orderBy("year", "name")

display(q4)

## Question 4

### Logic

To determine the youngest and oldest driver in each race, I combine the `results`, `drivers`, and `races` datasets. I define age as the driver's completed age in years on the date of the race. After calculating age, I use window functions to rank drivers within each race based on their age.



In [0]:
position_col = "positionOrder" if "positionOrder" in results.columns else "position"

podium = (
    results
    .join(races.select("raceId", "date"), "raceId")
    .withColumn("win", F.when(F.col(position_col) == 1, 1).otherwise(0))
    .withColumn("second", F.when(F.col(position_col) == 2, 1).otherwise(0))
    .withColumn("third", F.when(F.col(position_col) == 3, 1).otherwise(0))
)

window_spec = (
    Window
    .partitionBy("driverId")
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

q5 = (
    podium
    .withColumn("wins_so_far", F.sum("win").over(window_spec))
    .withColumn("seconds_so_far", F.sum("second").over(window_spec))
    .withColumn("thirds_so_far", F.sum("third").over(window_spec))
    .join(drivers.select("driverId", "forename", "surname"), "driverId")
)

display(q5)

## Question 5


To determine the number of podium finishes each driver has at any given race, I use the `results` dataset. I create three indicator variables to represent whether a driver finished in 1st, 2nd, or 3rd place in each race. Then, I use a cumulative window function to calculate running totals of these finishes for each driver over time.



In [0]:
pit_counts = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.count("*").alias("num_pit_stops"))
)

q6 = (
    pit_counts
    .groupBy("raceId")
    .agg(F.avg("num_pit_stops").alias("avg_pit_per_driver"))
    .join(races.select("raceId", "year", "name"), "raceId")
    .orderBy(F.col("avg_pit_per_driver").desc())
)

display(q6)

## Question 6

### My Question

Which race had the highest average number of pit stops per driver?


To answer this question, I calculate how many pit stops each driver made in each race using the `pit_stops` dataset. Then, I compute the average number of pit stops per driver within each race. Finally, I join the `races` dataset to include race information and sort the results in descending order.
